In [1]:
from pathlib import Path

from bennu_feature_extractor.environment_tools.fs_environment import \
    FSEnvironment
from bennu_feature_extractor_BoulderNet.Best_model_downloader import \
    BestModelDownloader
from bennu_feature_extractor_PDS.PDS_downloader import PDSDownloader
from bennu_feature_extractor_PDS.PDS_to_PNG import PDS_to_PNG
from graphviz import Source
from prefect import flow
from prefect.filesystems import LocalFileSystem
from prefect.futures import PrefectFuture, wait
from prefect.task_runners import ThreadPoolTaskRunner

c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\pydantic_settings\main.py:425: UserWarning: Config key `pyproject_toml_table_header` is set in model_config but will be ignored because no PyprojectTomlConfigSettingsSource source is configured. To use this config key, add a PyprojectTomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  self._settings_warn_unused_config_keys(sources, self.model_config)
c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\pydantic_settings\main.py:425: UserWarning: Config key `toml_file` is set in model_config but will be ignored because no TomlConfigSettingsSource source is configured. To use this config key, add a TomlConfigSettingsSource source to the settings sources via the settings_customise_sources hook.
  self._settings_warn_unused_config_keys(sources, self.model_config)
c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\tqdm_joblib\__init__

# Define paths

In [2]:
run_dir_store = LocalFileSystem.load("run-dir-storage")
dataDownloadPath = Path(r"F:\AO33_DATA")
spice_download_path : Path = Path(r"F:\AO33_SPICE")

@flow(task_runner=ThreadPoolTaskRunner(max_workers=20))
def data_loader_flow() -> FSEnvironment:
    tasks: list[PrefectFuture[FSEnvironment]] = []

    urls_to_download = [
    "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_data_calibrated_detailed_survey.zip",
    "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_data_reduced_detailed_survey.zip",
    "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_data_calibrated_orbit_b.zip",
    "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_data_calibrated_recon.zip",
    # "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_metadata.zip",
    # "https://sbnarchive.psi.edu/pds4/orex/downloads_ocams/ocams_calibration.zip"
    ]

    tasks.append(
        BestModelDownloader(
            run_dir_store,
            dataDownloadPath.as_posix(),
            Url="https://zenodo.org/records/8171052/files/best_model.zip?download=1"
        ).get_task_no_cache.submit(FSEnvironment.empty())
    )

    tasks += [
        PDSDownloader(
            run_dir_store,
            dataDownloadPath.as_posix(),
            Url=url
        ).get_task_no_cache.submit(FSEnvironment.empty())
        for url in urls_to_download
    ]

    wait(tasks)
    envs: list[FSEnvironment] = [f.result() for f in tasks]

    returned_env: FSEnvironment = FSEnvironment.merge(envs)

    return returned_env

# env = data_loader_flow()


## Define kernals

In [3]:
from pathlib import Path
from prefect import flow
from prefect.task_runners import ThreadPoolTaskRunner
from prefect.futures import PrefectFuture

from bennu_feature_extractor.environment_tools.fs_environment import FSEnvironment
from bennu_feature_extractor_PDS.SPICE_kernels_downloader import SPICEKernelGrabber

# Published 2019 metakernel (covers Detailed Survey, Orbit B, Recon in 2019)
MK_URLS = [
    "https://naif.jpl.nasa.gov/pub/naif/pds/pds4/orex/orex_spice/spice_kernels/mk/orx_2019_v08.tm",
]

# Optional: add a Bennu DSK as a plain file (no extraction)
EXTRA_URLS = [
    # "https://naif.jpl.nasa.gov/pub/naif/pds/pds4/orex/orex_spice/spice_kernels/dsk/bennu_g_00880mm_alt_obj_0000n00000_v021a.bds"
]

@flow(task_runner=ThreadPoolTaskRunner(max_workers=20))
def spice_kernals_loader_flow(
    result_storage,
    dataDownloadPath: Path,
) -> FSEnvironment:
    # One task that mirrors everything referenced by the MK(s)
    fut: PrefectFuture[FSEnvironment] = SPICEKernelGrabber(
        result_storage=result_storage,
        DownloadPath=dataDownloadPath.as_posix(),
        MkUrls=MK_URLS,          # <-- pass list of MK URLs here
        ExtraUrls=EXTRA_URLS,    # <-- optional
    ).get_task_no_cache.submit(FSEnvironment.empty())

    return fut.result()

# call it and merge into your pipeline env
spice_env = spice_kernals_loader_flow(run_dir_store, spice_download_path)
# env = FSEnvironment.merge([spice_env, env])


c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'default' attribute with value 'UTC' was provided to the `Field()` function, which has no effect in the context it was used. 'default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


23:23:57.151 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8627
See https://docs.prefect.io/3.0/manage/self-host#self-host-a-prefect-server for more information on running a dedicated Prefect server.

23:24:02.030 | INFO    | Flow run 'quixotic-falcon' - Beginning flow run 'quixotic-falcon' for flow 'spice-kernals-loader-flow'

23:24:02.627 | INFO    | Task run '_step_task_no_cache-726' - Downloading MK: https://naif.jpl.nasa.gov/pub/naif/pds/pds4/orex/orex_spice/spice_kernels/mk/orx_2019_v08.tm

23:24:03.820 | INFO    | Task run '_step_task_no_cache-726' - Found 1075 kernels in MK: https://naif.jpl.nasa.gov/pub/naif/pds/pds4/orex/orex_spice/spice_kernels/mk/orx_2019_v08.tm

...

23:49:29.088 | ERROR   | Task run '_step_task_no_cache-726' - Task run failed with exception: PermissionError(13, 'The process cannot access the file because it is being used by another process') - Retries are exhausted
joblib.externals.loky.process_executor._RemoteTraceback: 
"""
Traceback (most recent call last):
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py", line 490, in _process_worker
    r = call_item()
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py", line 291, in __call__
    return self.fn(*self.args, **self.kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 607, in __call__
    return [func(*args, **kwargs) for func, args, kwargs in self.items]
            ~~~~^^^^^^^^^^^^^^^^^
  File "C:\Users\Joshu\Documents\AO33\packages\bennu-feature-extractor-PDS\src\bennu_feature_extractor_PDS\SPICE_kernels_downloader.py", line 226, in __call__
    finally:
        ^^^^
  File "C:\Users\Joshu\AppData\Local\Programs\Python\Python313\Lib\pathlib\_local.py", line 780, in replace
    os.replace(self, target)
    ~~~~~~~~~~^^^^^^^^^^^^^^
PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'F:\\AO33_SPICE\\ck\\orx_ola_190703_scil2id04023_v02.bc.part' -> 'F:\\AO33_SPICE\\ck\\orx_ola_190703_scil2id04023_v02.bc'
"""

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\prefect\task_engine.py", line 809, in run_context
    yield self
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\prefect\task_engine.py", line 1382, in run_task_sync
    engine.call_task_fn(txn)
    ~~~~~~~~~~~~~~~~~~~^^^^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\prefect\task_engine.py", line 826, in call_task_fn
    result = call_with_parameters(self.task.fn, parameters)
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\prefect\utilities\callables.py", line 210, in call_with_parameters
    return fn(*args, **kwargs)
  File "C:\Users\Joshu\Documents\AO33\packages\bennu-feature-extractor\src\bennu_feature_extractor\step_base.py", line 34, in _step_task_no_cache
    return self.run(env)
           ~~~~~~~~^^^^^
  File "C:\Users\Joshu\Documents\AO33\packages\bennu-feature-extractor-PDS\src\bennu_feature_extractor_PDS\SPICE_kernels_downloader.py", line 334, in run
    else:
        ^
        self.logger.info(
        ^^^^^^^^^^^^^^^^^
            "Nothing to download — everything already present.")
    
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\tqdm_joblib\__init__.py", line 41, in __call__
    return super().__call__(it)
           ~~~~~~~~~~~~~~~~^^^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 2072, in __call__
    return output if self.return_generator else list(output)
                                                ~~~~^^^^^^^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 1682, in _get_outputs
    yield from self._retrieve()
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 1784, in _retrieve
    self._raise_error_fast()
    ~~~~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 1859, in _raise_error_fast
    error_job.get_result(self.timeout)
    ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "c:\Users\Joshu\Documents\AO33\integration-tests\.venv\Lib\site-packages\joblib\parallel.py", line 758, in get_result
    return self._return_or_raise()
 

23:49:29.197 | ERROR   | Task run '_step_task_no_cache-726' - Finished in state Failed("Task run encountered an exception PermissionError: [WinError None] The process cannot access the file because it is being used by another process: 'F:\\\\AO33_SPICE\\\\ck\\\\orx_ola_190703_scil2id04023_v02.bc.part' -> 'F:\\\\AO33_SPICE\\\\ck\\\\orx_ola_190703_scil2id04023_v02.bc'")

23:49:29.210 | ERROR   | Flow run 'quixotic-falcon' - Crash detected! Execution was aborted by an interrupt signal.

23:49:29.415 | INFO    | Flow run 'quixotic-falcon' - Finished in state Crashed('Execution was aborted by an interrupt signal.')

KeyboardInterrupt: 